In [ ]:
%cd ..

In [ ]:
import altair as alt
import pandas as pd

from notebooks.utils import load
from two_tower_confounding.simulation.simulator import get_position_bias

In [ ]:
df = load("3-expert-policy", "test.csv")
df.groupby(["policy_strength", "policy_temperature", "use_propensity_weighting"]).size(), len(df)

In [ ]:
def theme():
    return {
        "config": {
             "title": {
                "font": "serif",
                "fontWeight": "normal",
                "fontSize": 16,
                "dx": 5,
                 "subtitleFont": "serif",
                 "subtitleFontSize": 14,
            },
            "axis": {
                "titleFont": "serif",
                "titleFontWeight": "normal",
                "titleFontSize": 16,
                "labelFont": "serif",
                "labelFontWeight": "normal",
                "labelFontSize": 14
            },
            "headerColumn": {
                "titleFont": "serif",
                "titleFontWeight": "normal",
                "titleFontSize": 16,
                "labelFont": "serif",
                "labelFontWeight": "normal",
                "labelFontSize": 16
            },
            "text": {
                "font": "serif",
                "fontSize": 14,
            },
            "legend": {
                "symbolOpacity": 1,
                "titleFont": "serif",
                "titleFontWeight": "normal",
                "titleFontSize": 16,
                "labelFont": "serif",
                "labelFontWeight": "normal",
                "labelFontSize": 16,
            }
        },
    }

alt.themes.register("latex", theme)
alt.themes.enable("latex")

In [ ]:
def plot_bias(experiment, policy_strength, use_propensity_weighting, title="", subtitle="", width=225, height=156, color="blues", show_axis_title=True):
    scheme = {
        "blue": ["#C6DBEF", "#9ECAE1", "#6BAED6", "#3182BD", "#636363"],
        "green": ["#C7E9C0", "#A1D99B", "#74C476", "#31A354", "#636363"],
        "orange": ["#FDD0A2", "#fdae6b", "#FD8D3C", "#E6550E", "#636363"],
    }[color]

    
    bias_df = load(experiment, "bias.csv")
    bias_df = bias_df[bias_df["policy_strength"] == policy_strength]
    bias_df = bias_df[bias_df["use_propensity_weighting"] == use_propensity_weighting]

    
    n_positions = bias_df.position.max() + 1
    true_bias_df = pd.DataFrame(
        {
            "position": range(n_positions),
            "examination": get_position_bias(n_positions, 1),
            "policy_temperature": "Simulated Bias"
        }
    )

    bias_df = pd.concat([bias_df, true_bias_df])
    bias_df["position"] += 1

    base = alt.Chart(bias_df, width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)).encode(
        x=alt.X("position:Q", title="Position").axis(labelAngle=0, values=[1, 5, 10, 15, 20, 25]).scale(nice=False)
    )
    
    line = base.mark_line(point=True, opacity=1).encode(
        y=alt.Y("mean(examination):Q", title="Bias Logits" if show_axis_title else None).scale(domain=(-5, 0), clamp=True),
        color=alt.Color("policy_temperature:O", title="Temperature 𝜏").scale(range=scheme)
    )
    
    ci = base.mark_errorband(extent="ci").encode(
        y=alt.Y("examination:Q", title="Bias Logits" if show_axis_title else None).scale(domain=(-5, 0), clamp=True),
        color=alt.Color("policy_temperature:O").scale(range=scheme)
    )
    
    return ci + line



misfit_linear = plot_bias("3-model-misfit-linear", policy_strength=1, use_propensity_weighting=False, color="orange", title="Incorrect Relevance Model", subtitle="non-linear rel., linear relevance tower")
omitted_features = plot_bias("3-model-misfit-omitted-features", policy_strength=1, use_propensity_weighting=False, color="orange", title="Ommitted Variables", subtitle="non-linear rel., deep relevance tower", show_axis_title=False)
expert_policy = plot_bias("3-expert-policy", policy_strength=1, use_propensity_weighting=False, color="orange", title="Expert Policy", subtitle="expert rel., deep relevance tower", show_axis_title=False)


chart = (
    misfit_linear | omitted_features | expert_policy
).configure_line(
    size=2
).configure_point(
    size=15
).resolve_scale(
    y="shared"
).configure_concat(
    spacing=5
)

chart

In [ ]:
def plot_relevance(experiment, title="", subtitle="", width=225, height=125, color="blues", show_axis_title=True, use_propensity_weighting=False):
    scheme = {
        "blue": ["#C6DBEF", "#9ECAE1", "#6BAED6", "#3182BD", "#636363"],
        "green": ["#C7E9C0", "#A1D99B", "#74C476", "#31A354", "#636363"],
        "orange": ["#FDD0A2", "#FD8D3C", "#FD8D3C", "#E6550E", "#636363"],
    }[color]

    df = load(experiment, "test.csv")
    df = df[df["use_propensity_weighting"] == use_propensity_weighting]

    base = alt.Chart(df, width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)).encode(
        x=alt.X("policy_strength:O", title="Policy Strength 𝛼").axis(labelAngle=0)
    )
    
    line = base.mark_line(point=True, opacity=1).encode(
        y=alt.Y("mean(ndcg@10):Q", title="nDCG@10" if show_axis_title else None).scale(zero=False),
        color=alt.Color("policy_temperature:O", title="Temperature 𝜏").scale(range=scheme)
    )
    
    ci = base.mark_errorband(extent="ci").encode(
        y=alt.Y("ndcg@10:Q", title="nDCG@10" if show_axis_title else None).scale(zero=False),
        color=alt.Color("policy_temperature:O").scale(range=scheme)
    )
    
    return ci + line

misfit_linear = plot_relevance("3-model-misfit-linear", color="orange", title="Incorrect Relevance Model", subtitle="non-linear rel., linear relevance tower")
omitted_features = plot_relevance("3-model-misfit-omitted-features", color="orange", title="Ommitted Variables", subtitle="non-linear rel., deep relevance tower", show_axis_title=False)
expert_policy = plot_relevance("3-expert-policy", color="orange", title="Expert Policy", subtitle="expert rel., deep relevance tower", show_axis_title=False)


chart = (
    misfit_linear | omitted_features | expert_policy
).configure_line(
    size=2
).configure_point(
    size=15
).configure_concat(
    spacing=5
)

chart

In [ ]:
def plot_bias(experiment, policy_strength, use_propensity_weighting, title="", subtitle="", width=225, height=156, color="blues", show_axis_title=True):
    scheme = {
        "blue": ["#C6DBEF", "#9ECAE1", "#6BAED6", "#3182BD", "#636363"],
        "green": ["#C7E9C0", "#A1D99B", "#74C476", "#31A354", "#636363"],
        "orange": ["#FDD0A2", "#FD8D3C", "#FD8D3C", "#E6550E", "#636363"],
    }[color]

    
    bias_df = load(experiment, "bias.csv")
    bias_df = bias_df[bias_df["policy_strength"] == policy_strength]
    bias_df = bias_df[bias_df["use_propensity_weighting"] == use_propensity_weighting]

    
    n_positions = bias_df.position.max() + 1
    true_bias_df = pd.DataFrame(
        {
            "position": range(n_positions),
            "examination": get_position_bias(n_positions, 1),
            "policy_temperature": "Simulated Bias"
        }
    )

    bias_df = pd.concat([bias_df, true_bias_df])
    bias_df["position"] += 1

    base = alt.Chart(bias_df, width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)).encode(
        x=alt.X("position:Q", title="Position").axis(labelAngle=0, values=[1, 5, 10, 15, 20, 25]).scale(nice=False)
    )
    
    line = base.mark_line(point=True, opacity=1).encode(
        y=alt.Y("mean(examination):Q", title="Bias Logits" if show_axis_title else None).scale(domain=(-5, 0), clamp=True),
        color=alt.Color("policy_temperature:O", title="Temperature 𝜏").scale(range=scheme)
    )
    
    ci = base.mark_errorband(extent="ci").encode(
        y=alt.Y("examination:Q", title="Bias Logits" if show_axis_title else None).scale(domain=(-5, 0), clamp=True),
        color=alt.Color("policy_temperature:O").scale(range=scheme)
    )
    
    return ci + line



misfit_linear = plot_bias("3-model-misfit-linear", policy_strength=1, use_propensity_weighting=True, color="blue", title="Incorrect Relevance Model + IPS", subtitle="non-linear rel., linear relevance tower")
omitted_features = plot_bias("3-model-misfit-omitted-features", policy_strength=1, use_propensity_weighting=True, color="blue", title="Ommitted Variables + IPS", subtitle="non-linear relevance, deep relevance tower", show_axis_title=False)
expert_policy = plot_bias("3-expert-policy", policy_strength=1, use_propensity_weighting=True, color="blue", title="Expert Policy + IPS", subtitle="non-linear relevance, deep relevance tower", show_axis_title=False)


chart = (
    misfit_linear | omitted_features | expert_policy
).configure_line(
    size=2
).configure_point(
    size=15
).resolve_scale(
    y="shared"
).configure_concat(
    spacing=5
)

chart

In [ ]:
def plot_relevance(experiment, title="", subtitle="", width=225, height=125, color="blues", show_axis_title=True, use_propensity_weighting=True):
    scheme = {
        "blue": ["#C6DBEF", "#9ECAE1", "#6BAED6", "#3182BD", "#636363"],
        "green": ["#C7E9C0", "#A1D99B", "#74C476", "#31A354", "#636363"],
        "orange": ["#FDD0A2", "#FD8D3C", "#FD8D3C", "#E6550E", "#636363"],
    }[color]

    df = load(experiment, "test.csv")
    df = df[df["use_propensity_weighting"] == use_propensity_weighting]

    base = alt.Chart(df, width=width, height=height, title=alt.Title(text=title, subtitle=subtitle)).encode(
        x=alt.X("policy_strength:O", title="Policy Strength 𝛼").axis(labelAngle=0)
    )
    
    line = base.mark_line(point=True, opacity=1).encode(
        y=alt.Y("mean(ndcg@10):Q", title="nDCG@10" if show_axis_title else None).scale(zero=False),
        color=alt.Color("policy_temperature:O", title="Temperature 𝜏").scale(range=scheme)
    )
    
    ci = base.mark_errorband(extent="ci").encode(
        y=alt.Y("ndcg@10:Q", title="nDCG@10" if show_axis_title else None).scale(zero=False),
        color=alt.Color("policy_temperature:O").scale(range=scheme)
    )
    
    return ci + line

misfit_linear = plot_relevance("3-model-misfit-linear", color="blue", title="Incorrect Relevance Model + IPS", subtitle="non-linear rel., linear relevance tower")
omitted_features = plot_relevance("3-model-misfit-omitted-features", color="blue", title="Ommitted Variables + IPS", subtitle="non-linear relevance, deep relevance tower", show_axis_title=False)
expert_policy = plot_relevance("3-expert-policy", color="blue", title="Expert Policy + IPS", subtitle="non-linear relevance, deep relevance tower", show_axis_title=False)


chart = (
    misfit_linear | omitted_features | expert_policy
).configure_line(
    size=2
).configure_point(
    size=15
).configure_concat(
    spacing=5
)

chart